# Build Hybrid Index (Dense + Facet-Vector) for Trait-Aligned RAG

This notebook extends `build_dual_index.ipynb` with a **structured 30-d facet vector** built from each profile's facet signals (high/mod/low/none/n/e). Cosine similarity in this space is *trait-aligned by construction* — the dimensions ARE the NEO-PI-R facets, with high/low encoded at opposite ends. We expect this to beat the dense-only retriever, especially for Conscientiousness and Neuroticism.

Pipeline:
1. Build dual-embedding FAISS index (raw + profile) AND save parallel `facet_vectors.npy`.
2. Encode val essays (dense + facet).
3. Sanity check retrieval-acc with: dense-only, facet-only, hybrid (β·dense + γ·facet).
4. Per-trait (β, γ) sweep on val. The winning weights become the defaults in `rag/retriever.py`.

**Anti-leakage:** Test set is not embedded here. Val set is used to tune β/γ only.

**Output:** `data/vector_db/essays_dual/{vectors.faiss, vectors_meta.jsonl, facet_vectors.npy}`

## 1. CONFIG

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = str(Path(os.getcwd()).parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

TRAIN_CSV          = os.path.join(PROJECT_ROOT, 'data', 'split', 'essays', 'train.csv')
VAL_CSV            = os.path.join(PROJECT_ROOT, 'data', 'split', 'essays', 'val.csv')
PROFILE_STORE_PATH = os.path.join(PROJECT_ROOT, 'data', 'profile_db', 'essays', 'profile_store.jsonl')
VAL_PROFILE_PATH   = os.path.join(PROJECT_ROOT, 'data', 'profile_db', 'essays_test', 'profile_store.jsonl')

FINETUNED_MODEL_DIR = os.path.join(PROJECT_ROOT, 'models', 'sbert_essays_finetuned')
BASE_MODEL          = 'nomic-ai/nomic-embed-text-v1.5'
MAX_SEQ_LENGTH      = 2048
ALPHA               = 0.5  # raw vs profile fusion weight inside the dense vector

OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'vector_db', 'essays_dual')
TOP_K      = 5

TRAITS      = ['cOPN', 'cCON', 'cEXT', 'cAGR', 'cNEU']
TRAIT_NAMES = {'cOPN': 'Openness', 'cCON': 'Conscientiousness', 'cEXT': 'Extraversion',
               'cAGR': 'Agreeableness', 'cNEU': 'Neuroticism'}
TRAIT_FULL  = {'cOPN': 'Openness to Experience', 'cCON': 'Conscientiousness', 'cEXT': 'Extraversion',
               'cAGR': 'Agreeableness', 'cNEU': 'Neuroticism'}

for label, p in [('TRAIN_CSV', TRAIN_CSV), ('VAL_CSV', VAL_CSV),
                 ('PROFILE_STORE', PROFILE_STORE_PATH), ('VAL_PROFILES', VAL_PROFILE_PATH)]:
    print(f'  {label:18s}: {"OK" if Path(p).exists() else "MISSING":7s}  {p}')
print(f'  OUTPUT_DIR        : {OUTPUT_DIR}')


## 2. Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import faiss
from sklearn.metrics import accuracy_score, f1_score

import rag.embedder as _embedder
_embedder.FINETUNED_MODEL_DIR = FINETUNED_MODEL_DIR
_embedder.MAX_SEQ_LENGTH      = MAX_SEQ_LENGTH
_embedder.ALPHA               = ALPHA

from rag.profiler.store   import ProfileStore
from rag.profiler.prompts import FACETS
from rag.embedder         import get_dual_embedding, get_embedding
from rag.facet_vector     import (
    facet_vector,
    build_facet_matrix,
    facet_cosine,
    hybrid_score,
    FACET_ORDER,
    TRAIT_FACET_SLICES,
    save_facet_matrix,
    load_facet_matrix,
)
print('Imports OK')


## 3. Build dense index + facet matrix

Uses the upgraded `build_features.build_index` which now also writes `facet_vectors.npy` parallel to `vectors.faiss`.

In [ ]:
from rag.runners.build_features import build_index
train_df = pd.read_csv(TRAIN_CSV)

build_index(
    data               = train_df,
    profile_store_path = PROFILE_STORE_PATH,
    output_dir         = OUTPUT_DIR,
    alpha              = ALPHA,
    force_rebuild      = False,  # set True if facet_vectors.npy is missing
)

print('\nIndex files:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    print(f'  {fname:30s} {size_mb:>8.2f} MB')


## 4. Load index + val embeddings + val facet matrix

In [ ]:
# Train side
train_index   = faiss.read_index(os.path.join(OUTPUT_DIR, 'vectors.faiss'))
train_facets  = load_facet_matrix(os.path.join(OUTPUT_DIR, 'facet_vectors.npy'))
train_meta    = []
with open(os.path.join(OUTPUT_DIR, 'vectors_meta.jsonl'), encoding='utf-8') as f:
    for line in f:
        train_meta.append(json.loads(line))
print(f'Train: {train_index.ntotal} vectors, facet matrix {train_facets.shape}')

# Val side
val_df    = pd.read_csv(VAL_CSV)
val_store = ProfileStore(VAL_PROFILE_PATH)
val_store.load()
print(f'Val: {len(val_df)} essays, profiles available: {len(val_store)}')


In [ ]:
def _profile_to_full_text(profile):
    facets = profile.get('facets', {})
    ling   = profile.get('linguistic', {})
    lines = []
    for code, *_ in FACETS:
        f = facets.get(code)
        if f and f.get('signal'):
            lines.append(f"{code} | {f['signal']} | {f.get('evidence','')}")
    for k, v in ling.items():
        if v:
            lines.append(f'{k}: {v}')
    return '\n'.join(lines)

print('Encoding val essays (dense + facet) ...')
val_dense_embs = []
val_facet_rows = []
for i, row in val_df.iterrows():
    raw = str(row['text'])
    entry = val_store.get(f'user_{i}')
    if entry and entry.get('valid'):
        ptxt = _profile_to_full_text(entry)
        d = get_dual_embedding(raw, ptxt, alpha=ALPHA)
        f = facet_vector(entry, normalize=True)
    else:
        d = get_embedding(raw)
        f = np.zeros(len(FACET_ORDER), dtype=np.float32)
    val_dense_embs.append(d)
    val_facet_rows.append(f)
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(val_df)}')

val_dense_embs = np.array(val_dense_embs, dtype='float32')
val_facet_rows = np.array(val_facet_rows, dtype='float32')
print(f'val_dense_embs={val_dense_embs.shape}  val_facet_rows={val_facet_rows.shape}')


## 5. Three retrieval strategies — head-to-head

**Dense-only**, **facet-only** (full 30-d), and **hybrid**. Facet-only is the cleanest test of whether the structured trait vector carries real signal.

In [ ]:
def retrieve_topk(scores_per_query, top_k):
    """Return (N, K) integer matrix of top-K indices given an (N, M) score matrix."""
    return np.argsort(-scores_per_query, axis=1)[:, :top_k]

def vote_acc(neighbour_idx, val_labels, train_meta, trait_full):
    """Top-K majority-vote accuracy."""
    preds = []
    for nbrs in neighbour_idx:
        nbr_labels = [train_meta[j]['trait_labels'].get(trait_full, 'low') for j in nbrs]
        preds.append('high' if nbr_labels.count('high') > nbr_labels.count('low') else 'low')
    return accuracy_score(val_labels, preds), f1_score(val_labels, preds, average='macro')

def normalise_val_labels(col):
    out = []
    for v in col:
        s = str(v).strip().lower()
        if s in ('1', '1.0'): s = 'high'
        elif s in ('0', '0.0'): s = 'low'
        out.append(s)
    return out

# Dense scores: -L2 distance (higher = more similar). Use raw faiss for indices.
D_l2, I_dense = train_index.search(val_dense_embs, train_index.ntotal)
dense_scores  = -D_l2  # (N_val, N_train) — but indices are NOT identity, so we must permute.
# Build a (N_val, N_train) score matrix in row-aligned order.
dense_full = np.full((val_dense_embs.shape[0], train_index.ntotal), -np.inf, dtype=np.float32)
for r in range(val_dense_embs.shape[0]):
    dense_full[r, I_dense[r]] = -D_l2[r]

# Facet scores: cosine on the full 30-d vector (broadcasted matmul).
# Both sides are L2-normed already.
facet_full = val_facet_rows @ train_facets.T  # (N_val, N_train)

# Min-max rescale dense per row to ~[-1, 1] so beta/gamma fusion is meaningful.
def per_row_minmax(M):
    M = M.copy()
    finite = np.isfinite(M)
    M[~finite] = 0.0
    mins = M.min(axis=1, keepdims=True)
    maxs = M.max(axis=1, keepdims=True)
    rng  = np.where(maxs - mins > 0, maxs - mins, 1.0)
    return 2.0 * (M - mins) / rng - 1.0

dense_scaled = per_row_minmax(dense_full)
print('dense_scaled', dense_scaled.shape, 'facet_full', facet_full.shape)


In [ ]:
print(f"\n=== Retrieval-based label accuracy (top-{TOP_K} majority vote) ===\n")
print(f"{'Trait':22s} {'Dense':>9s}  {'Facet':>9s}  {'Hybrid':>9s}  {'F1 hyb':>9s}")
print('-' * 72)

for trait_col, trait_full in TRAIT_FULL.items():
    val_labels = normalise_val_labels(val_df[trait_col].tolist())

    nbr_dense  = retrieve_topk(dense_scaled, TOP_K)
    nbr_facet  = retrieve_topk(facet_full,   TOP_K)
    # 50/50 fusion as a starting point; tune in next cell.
    fused      = 0.5 * dense_scaled + 0.5 * facet_full
    nbr_hybrid = retrieve_topk(fused, TOP_K)

    a_d, _   = vote_acc(nbr_dense,  val_labels, train_meta, trait_full)
    a_f, _   = vote_acc(nbr_facet,  val_labels, train_meta, trait_full)
    a_h, f_h = vote_acc(nbr_hybrid, val_labels, train_meta, trait_full)

    print(f'  {TRAIT_NAMES[trait_col]:20s}  {a_d:>9.4f}  {a_f:>9.4f}  {a_h:>9.4f}  {f_h:>9.4f}')


## 6. Per-trait (β, γ) sweep

For each trait, scan a grid of (β, γ) and pick the pair maximising val accuracy. Also try **trait-restricted facet cosine** — masking the 30-d vector down to the 6 facets that load on the target trait. Expected to add 1–4pp because cross-trait noise vanishes.

In [ ]:
BETA_GRID  = [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
# gamma = 1 - beta for a clean 1-d sweep; you can switch to a 2-d grid if needed.

def trait_restricted_facet_scores(val_facets, train_facets, trait_code):
    """Cosine on the 6-facet sub-space loading on the trait."""
    idx = np.array(TRAIT_FACET_SLICES[trait_code], dtype=np.int64)
    Vq = val_facets[:, idx]
    Vt = train_facets[:, idx]
    # Re-normalise on the restricted axes.
    Vq = Vq / np.where(np.linalg.norm(Vq, axis=1, keepdims=True) > 0,
                       np.linalg.norm(Vq, axis=1, keepdims=True), 1.0)
    Vt = Vt / np.where(np.linalg.norm(Vt, axis=1, keepdims=True) > 0,
                       np.linalg.norm(Vt, axis=1, keepdims=True), 1.0)
    return Vq @ Vt.T

best_weights = {}
print(f"Per-trait (β, γ=1-β) sweep — TOP_K={TOP_K} — best val acc shown.\n")
print(f"{'Trait':22s} {'best β':>8s} {'best γ':>8s} {'best acc':>10s} {'restricted?':>13s}")
print('-' * 70)

for trait_col, trait_full in TRAIT_FULL.items():
    val_labels = normalise_val_labels(val_df[trait_col].tolist())
    facet_global    = facet_full
    facet_restricted = trait_restricted_facet_scores(val_facet_rows, train_facets, trait_col)

    best = (-1.0, None, None, False)  # (acc, beta, gamma, restricted)
    for restricted, F in (('global', facet_global), ('restricted', facet_restricted)):
        for beta in BETA_GRID:
            gamma = 1.0 - beta
            fused = beta * dense_scaled + gamma * F
            nbr   = retrieve_topk(fused, TOP_K)
            acc, _ = vote_acc(nbr, val_labels, train_meta, trait_full)
            if acc > best[0]:
                best = (acc, beta, gamma, restricted)
    acc, beta, gamma, restr = best
    best_weights[trait_full] = (beta, gamma, restr)
    print(f'  {TRAIT_NAMES[trait_col]:20s} {beta:>8.2f} {gamma:>8.2f} {acc:>10.4f} {restr:>13s}')

print('\nRecommended DEFAULT_TRAIT_WEIGHTS in rag/retriever.py:')
for k, (b, g, _) in best_weights.items():
    print(f'    "{k}": ({b:.2f}, {g:.2f}),')


## 7. Smoke test — hybrid retriever class returns sane neighbours

Confirms the production code path (`FeatureRAGRetriever(use_hybrid=True)`) works end-to-end.

In [ ]:
from rag.retriever import FeatureRAGRetriever

retriever = FeatureRAGRetriever(db_dir=OUTPUT_DIR, use_hybrid=True)
test_idx  = 0
row       = val_df.iloc[test_idx]
raw       = str(row['text'])
entry     = val_store.get(f'user_{test_idx}')
ptxt      = _profile_to_full_text(entry) if entry and entry.get('valid') else None

for tr_full in TRAIT_FULL.values():
    res = retriever.retrieve(
        posts=raw, trait=tr_full, top_k=3,
        profile_text=ptxt, query_profile_dict=entry,
    )
    print(f'\n{tr_full}:')
    for r in res:
        sc = r.get('scores', {})
        print(f'  {r["user_id"]}  label={r["label"]:5s}  fused={sc.get("fused", float("nan")):+.3f}  '
              f'dense={sc.get("dense", float("nan")):+.3f}  facet={sc.get("facet", float("nan")):+.3f}')
